<a href="https://colab.research.google.com/github/Aleem-mja/DeepLearning-Project-SLIIT-SE4050/blob/IT22313966/IMDB_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -------------------------------
# 1) Imports, seeds, and globals
# -------------------------------
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import os

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Hyperparameters / globals
MAX_FEATURES = 10000   # number of words to keep (most frequent)
MAXLEN = 500           # cut texts after this number of words (among top MAX_FEATURES)
EMBED_DIM = 128
RNN_UNITS = 64
BATCH_SIZE = 128
EPOCHS = 10
VAL_SAMPLES = 5000     # size of validation set taken from training set
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
# -----------------------------------------
# 2) Data loading and preprocessing (Keras)
# -----------------------------------------
print("Loading IMDB dataset (integer-encoded reviews)...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=MAX_FEATURES)

# Pad sequences so all inputs have the same length
x_train = pad_sequences(x_train, maxlen=MAXLEN)
x_test  = pad_sequences(x_test,  maxlen=MAXLEN)

print("Shapes after padding:")
print(f"  x_train: {x_train.shape}, y_train: {y_train.shape}")
print(f"  x_test:  {x_test.shape},  y_test:  {y_test.shape}")

# Create validation set from the end of training set
x_val = x_train[-VAL_SAMPLES:]
y_val = y_train[-VAL_SAMPLES:]
x_train = x_train[:-VAL_SAMPLES]
y_train = y_train[:-VAL_SAMPLES]

print("Shapes after creating validation set:")
print(f"  x_train: {x_train.shape}, x_val: {x_val.shape}, x_test: {x_test.shape}")


Loading IMDB dataset (integer-encoded reviews)...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Shapes after padding:
  x_train: (25000, 500), y_train: (25000,)
  x_test:  (25000, 500),  y_test:  (25000,)
Shapes after creating validation set:
  x_train: (20000, 500), x_val: (5000, 500), x_test: (25000, 500)


In [ ]:
# -----------------------------------------
# 3) Model building (function for reuse)
# -----------------------------------------
def build_rnn_model(max_features=MAX_FEATURES, embed_dim=EMBED_DIM, rnn_units=RNN_UNITS, maxlen=MAXLEN, dropout_rate=0.5):
    """
    Build and return a Sequential RNN model:
      Embedding -> SimpleRNN -> SimpleRNN -> Dropout -> Dense(sigmoid)
    """
    model = Sequential([
        Embedding(max_features, embed_dim, input_length=maxlen),
        SimpleRNN(rnn_units, return_sequences=True),
        SimpleRNN(rnn_units),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid')
    ])
    return model

# Build and compile
print("Building model...")
model = build_rnn_model()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


Building model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ---------------------------
# 4) Train the model
# ---------------------------
print("Training the model...")
history = model.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(x_val, y_val),
    verbose=1
)

# Save the trained model for later use
model_path = os.path.join(MODEL_DIR, "imdb_rnn_model.h5")
model.save(model_path)
print(f"Saved model to: {model_path}")


Training the model...
Epoch 1/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 94s 580ms/step - accuracy: 0.5377 - loss: 0.7151 - val_accuracy: 0.5618 - val_loss: 0.6735
Epoch 2/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 137s 546ms/step - accuracy: 0.5761 - loss: 0.6916 - val_accuracy: 0.6146 - val_loss: 0.6355
Epoch 3/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 80s 511ms/step - accuracy: 0.6992 - loss: 0.5672 - val_accuracy: 0.7916 - val_loss: 0.4652
Epoch 4/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 85s 534ms/step - accuracy: 0.8064 - loss: 0.4564 - val_accuracy: 0.8174 - val_loss: 0.4331
Epoch 5/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 79s 506ms/step - accuracy: 0.8476 - loss: 0.3743 - val_accuracy: 0.8040 - val_loss: 0.4595
Epoch 6/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 79s 503ms/step - accuracy: 0.8552 - loss: 0.3625 - val_accuracy: 0.8202 - val_loss: 0.4310
Epoch 7/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 80s 513ms/step - accuracy: 0.8824 - loss: 0.2979 - val_accuracy: 0.8150 - val_loss: 0.4479
Epoch 8/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 79s 505ms/step - acc

Saved model to: models/imdb_rnn_model.h5


In [ ]:
# -----------------------------------
# 5) Evaluate on test data & metrics
# -----------------------------------
print("Predicting on test set...")
y_pred_prob = model.predict(x_test, batch_size=BATCH_SIZE)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print(f"Test Accuracy : {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall   : {recall:.4f}")
print(f"Test F1 Score : {f1:.4f}")


In [ ]:
# -----------------------------------
# 6) Plot training & validation curves
# -----------------------------------
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------
# 7) Predict helper: raw text -> model prediction
# ------------------------------------------------
# Get the mapping word -> integer index
word_index = imdb.get_word_index()             # maps word -> integer (no special tokens)
# Keras reserves index 0 (padding), 1 (start), 2 (OOV) by convention for this dataset.
index_offset = 3

# Build reverse mapping if you want to decode sequences later
reverse_word_index = { (index + index_offset): word for word, index in word_index.items() }
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"

def text_to_sequence(text, word_index=word_index, maxlen=MAXLEN):
    """
    Simple tokenizer: lowercases and splits on whitespace.
    Unknown words map to index 2 (UNK).
    """
    words = text.lower().split()
    seq = []
    for w in words:
        # word_index returns 1-based indices in dataset mapping; we add +index_offset
        idx = word_index.get(w)
        if idx is not None:
            seq.append(idx + index_offset)
        else:
            seq.append(2)  # unk
    seq = pad_sequences([seq], maxlen=maxlen)
    return seq

def predict_sentiment(text):
    seq = text_to_sequence(text)
    prob = model.predict(seq)[0][0]
    label = "Positive" if prob > 0.5 else "Negative"
    return {"label": label, "probability": float(prob)}

# Example
example_review = "This movie was fantastic! The acting was superb and the plot kept me engaged throughout."
print("Example prediction:", predict_sentiment(example_review))
